In [0]:
# Databricks notebook source
# ==============================================================================
# PHASE 6: Secure Authentication & ERP Data Simulator (TPC-H -> ADLS Gen2)
# ==============================================================================

storage_account_name = "datalakeseniorproj012026"
kv_scope_name = "kv-portfolio12026"

print("1. Authenticating via Azure Key Vault Secrets...")
try:
    client_id = dbutils.secrets.get(scope=kv_scope_name, key="sp-client-id")
    tenant_id = dbutils.secrets.get(scope=kv_scope_name, key="sp-tenant-id")
    client_secret = dbutils.secrets.get(scope=kv_scope_name, key="sp-client-secret")
except Exception as e:
    print(f"Failed to retrieve secrets: {e}")
    raise

print("2. Injecting OAuth 2.0 configuration into Spark Session...")
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

raw_landing_uri = f"abfss://raw-landing@{storage_account_name}.dfs.core.windows.net/erp_export/"

print("3. Reading from Databricks built-in TPC-H dataset...")
df_lineitem = spark.read.table("samples.tpch.lineitem")
df_orders = spark.read.table("samples.tpch.orders")

print("4. Writing 'lineitem' table to ADLS raw-landing zone...")
df_lineitem.write.mode("overwrite").format("parquet").save(raw_landing_uri + "lineitem/")

print("5. Writing 'orders' table to ADLS raw-landing zone...")
df_orders.write.mode("overwrite").format("parquet").save(raw_landing_uri + "orders/")

print("==============================================================================")
print("SUCCESS: Raw simulated ERP data successfully landed in ADLS Gen2!")
print("==============================================================================")

display(dbutils.fs.ls(raw_landing_uri))